In [ ]:
# [BLOCK 1] CÀI ĐẶT THƯ VIỆN & KẾT NỐI API

# 1. Cài đặt chế độ ẩn (-q) các thư viện lõi: pinecone (lưu trữ vector), transformers (chạy mô hình AI), pandas (xử lý dữ liệu), google-genai (giao tiếp Gemini API), nltk và underthesea (xử lý ngôn ngữ tự nhiên), rapidfuzz (so khớp chuỗi), và deep-translator (dịch văn bản).
!pip install -q pinecone transformers pandas google-genai nltk underthesea rapidfuzz deep-translator

# 2. Import các thư viện hệ thống và xử lý mảng/bảng dữ liệu cơ bản
import os, re, csv, shutil, json, time
import pandas as pd
import numpy as np

# 3. Import Pytorch (để chạy model SigLIP trên GPU) và PIL (để mở, xử lý hình ảnh)
import torch
from PIL import Image

# 4. Cài đặt các công cụ tối ưu tốc độ: lru_cache (Lưu nháp bộ nhớ tạm) và ThreadPoolExecutor (Chạy đa luồng song song)
from functools import lru_cache
from concurrent.futures import ThreadPoolExecutor, as_completed

# 5. Import SDK của các nền tảng AI đám mây (Google Gemini, Pinecone, HuggingFace)
from google import genai
from google.genai import types
from pinecone import Pinecone
from transformers import AutoModel, AutoTokenizer
from deep_translator import GoogleTranslator

# 6. Công cụ tải file xuống máy tính từ Colab
from google.colab import files

# 7. Khai báo API Key để cấp quyền truy cập vào Pinecone và Gemini
PINECONE_API_KEY = "YOUR_PINECONE_API_KEY_HERE"
GEMINI_API_KEY = "YOUR_GEMINI_API_KEY_HERE"

# 8. Xin quyền và kết nối vào Google Drive (để code đọc được ảnh và file Map Keyframe)
from google.colab import drive
drive.mount('/content/drive')

# 9. Đăng nhập vào hệ thống Pinecone và trỏ thẳng vào dataset có tên "aic-26"
pc = Pinecone(api_key=PINECONE_API_KEY)
index = pc.Index("aic-26")

# 10. Khởi tạo client kết nối với Gemini API để chuẩn bị gọi mô hình Ngôn ngữ - Hình ảnh (VLM) cho bước xác thực.
client = genai.Client(api_key=GEMINI_API_KEY)

# 11. Tự động kiểm tra môi trường thực thi: thiết lập thiết bị xử lý là GPU ("cuda") nếu hệ thống có hỗ trợ, ngược lại sẽ sử dụng CPU ("cpu").
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
# [BLOCK 2] NẠP MODEL LÊN GPU & HỆ THỐNG MAP KEYFRAME

# 1. Khai báo định danh của mô hình SigLIP (mô hình ngôn ngữ - hình ảnh đa phương thức) trên HuggingFace.
MODEL_NAME = "google/siglip-so400m-patch14-384"

# 2. Khởi tạo tokenizer để xử lý và mã hóa dữ liệu văn bản đầu vào thành các token.
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

# 3. Tải trọng số mô hình SigLIP vào bộ nhớ.
# Tự động ép kiểu dữ liệu về float16 (FP16) nếu sử dụng GPU để tối ưu VRAM và tăng tốc độ suy luận, hoặc float32 nếu dùng CPU.
# Phương thức .eval() đặt mô hình vào chế độ suy luận (tắt các chức năng training như Dropout).
model = AutoModel.from_pretrained(
    MODEL_NAME,
    torch_dtype=torch.float16 if DEVICE.type == 'cuda' else torch.float32
).text_model.to(DEVICE).eval()

# 4. Khởi tạo công cụ dịch thuật để chuyển đổi câu truy vấn từ tiếng Việt sang tiếng Anh nhằm tối ưu độ chính xác của mô hình SigLIP.
translator = GoogleTranslator(source='vi', target='en')

# 5. Khai báo đường dẫn trỏ tới thư mục chứa các tệp CSV ánh xạ khung hình (Map Keyframe) trên Google Drive.
MAP_DIR = "/content/drive/MyDrive/AIC26/mapkeyframes"

# 6. Khởi tạo và kiểm tra từ điển toàn cục (global dictionary) 'map_db'.
# Việc này giúp bộ nhớ đệm (cache) giữ lại dữ liệu ánh xạ, tránh tình trạng đọc lại file từ đầu mỗi khi thực thi lại block code.
if 'map_db' not in globals() or not map_db:
    map_db = {}

    if os.path.exists(MAP_DIR):
        # 7. Duyệt qua tất cả các tập tin trong thư mục, chỉ lọc và xử lý các tập tin có định dạng .csv
        for file_name in os.listdir(MAP_DIR):
            if not file_name.endswith('.csv'): continue

            # 8. Trích xuất mã định danh của video (Video ID) bằng cách loại bỏ đuôi '.csv' và các khoảng trắng thừa.
            vid = file_name.replace('.csv', '').strip().upper()

            try:
                # 9. Đọc tập tin CSV và kiểm tra tính toàn vẹn của dữ liệu thông qua hai cột bắt buộc: 'n' (số thứ tự keyframe) và 'frame_idx' (số khung hình gốc).
                df_map = pd.read_csv(os.path.join(MAP_DIR, file_name))
                if 'n' in df_map.columns and 'frame_idx' in df_map.columns:

                    # 10. Chuyển đổi cấu trúc bảng thành từ điển và lưu vào map_db với cấu trúc: map_db[video_id] = {n: frame_idx}
                    map_db[vid] = {int(row['n']): int(row['frame_idx']) for _, row in df_map.iterrows()}

            # 11. Bỏ qua các tập tin bị lỗi đọc/ghi hoặc sai cấu trúc mà không làm gián đoạn toàn bộ quá trình.
            except Exception as e:
                pass

Loading weights:   0%|          | 0/888 [00:00<?, ?it/s]

In [ ]:
# [BLOCK 3] LÕI THUẬT TOÁN (TRUY XUẤT VECTOR & QUY HOẠCH ĐỘNG)

# Sử dụng @lru_cache để lưu bộ nhớ đệm (tối đa 128 kết quả), giúp giảm độ trễ và tiết kiệm API quota khi gặp lại các truy vấn trùng lặp.
@lru_cache(maxsize=128)
def quest_rewrite(query_text):
    prompt = f"""Bạn là chuyên gia thị giác máy tính. Chuyển đổi truy vấn sau thành chuỗi từ khóa miêu tả hình ảnh (Tiếng Anh). Bỏ từ trừu tượng (đầu tiên, chuẩn bị). Trọng tâm vào vật thể chạm nhau. Truy vấn: '{query_text}'"""
    try:
        # Gọi mô hình LLM để tối ưu hóa câu truy vấn thành các từ khóa thị giác.
        res = client.models.generate_content(
            model='gemini-3.5-flash',
            contents=prompt,
            config=types.GenerateContentConfig(temperature=0.0)
        )
        return res.text.strip()
    except: return query_text

def aic_retrieval_engine(query_vi, use_quest=True, top_k_pinecone=1000, top_k_final=400):
    try: en_query = translator.translate(query_vi)
    except: en_query = query_vi

    visual_query = quest_rewrite(en_query) if use_quest else en_query

    # Mã hóa chuỗi văn bản và trích xuất đặc trưng vector (Text Embedding) thông qua mô hình SigLIP.
    inputs = tokenizer([visual_query], padding="max_length", max_length=64, truncation=True, return_tensors="pt").to(DEVICE)
    with torch.inference_mode():
        query_vector = model(**inputs).pooler_output[0].cpu().numpy().tolist()

    # Truy vấn độ tương đồng Cosine trên cơ sở dữ liệu Pinecone để tìm kiếm các vector không gian gần nhất.
    pc_results = index.query(vector=query_vector, top_k=top_k_pinecone, include_metadata=True)['matches']

    # Sắp xếp và định dạng lại kết quả trả về, đồng thời trích xuất ID video và chỉ số keyframe (n) từ siêu dữ liệu (metadata).
    return sorted([{"video_id": m.get('metadata', {}).get('video', '').strip().upper(), "n": int(''.join(filter(str.isdigit, m.get('metadata', {}).get('frame', '0'))) or 0), "score": m['score']} for m in pc_results], key=lambda x: x['score'], reverse=True)[:top_k_final]


# 3. Hàm bọc (Wrapper) để thực thi lệnh truy xuất cho một sự kiện đơn lẻ.
def process_single_event(i, event_text):
    results = aic_retrieval_engine(
        event_text,
        use_quest=False,
        top_k_pinecone=4000,
        top_k_final=2000
    )
    return i, results

# Tách các truy vấn có cấu trúc (ví dụ: E1:, E2:) thành một danh sách các sự kiện độc lập theo trình tự thời gian.
def split_trake_query(query_text):
    if re.search(r'E1|\(1\)|1\.', query_text, re.IGNORECASE):
        try:
            parts = re.split(r'E\d+\s*|\(\d+\)\s*|\d+\.\s*', query_text, flags=re.IGNORECASE)
            context = (parts[0].split(":")[0] + " " if ":" in parts[0] else parts[0]).strip() + " "
            events = [context + a.strip(' ,;.:\n') for a in parts[1:] if a.strip()]
            if len(events) > 1: return events
        except: pass
    return [query_text]

def dp_solve(query_id, query_text, top_videos=30, lambda_penalty=0.00005):
    sub_events = split_trake_query(query_text)
    N = len(sub_events)
    event_frame_scores, vid_votes = {i: {} for i in range(N)}, {}

    # Thực thi đa luồng (Multithreading) để truy xuất dữ liệu vector cho tất cả các sự kiện (N) cùng lúc.
    with ThreadPoolExecutor(max_workers=min(N, 5)) as executor:
        futures = {executor.submit(process_single_event, i, evt): i for i, evt in enumerate(sub_events)}
        for future in as_completed(futures):
            i, evt_results = future.result()
            for res in evt_results:
                vid, n_val, score = res['video_id'], res['n'], res['score']
                if vid not in event_frame_scores[i]: event_frame_scores[i][vid] = {}
                event_frame_scores[i][vid][n_val] = score
                if vid not in vid_votes: vid_votes[vid] = set()
                vid_votes[vid].add(i)

    # Chấm điểm tổng hợp và lọc ra danh sách các video tiềm năng (candidate_vids) nhằm giảm thiểu không gian tìm kiếm cho thuật toán DP.
    vid_total_scores = {vid: (len(v_found), sum(max(event_frame_scores[i][vid].values()) for i in range(N) if vid in event_frame_scores[i])) for vid, v_found in vid_votes.items()}
    candidate_vids = sorted(vid_total_scores.keys(), key=lambda v: (vid_total_scores[v][0], vid_total_scores[v][1]), reverse=True)[:top_videos]
    final_sequences = []

    # Vòng lặp Quy hoạch động cho từng video ứng viên.
    for vid in candidate_vids:
        T_v = sorted({k for i in range(N) if vid in event_frame_scores[i] for k in event_frame_scores[i][vid].keys()})
        if len(T_v) < N: continue
        K = len(T_v)

        # Khởi tạo ma trận chi phí (dp) và ma trận lưu vết (trace).
        dp, trace = np.full((N, K), -np.inf), np.full((N, K), -1, dtype=int)

        # Điều kiện cơ sở (Base case) cho sự kiện đầu tiên.
        for t_idx, t in enumerate(T_v): dp[0, t_idx] = event_frame_scores[0].get(vid, {}).get(t, 0.0)

        # Bước tiến (State transition) với kỹ thuật tối ưu hóa độ phức tạp thời gian bằng biến running_max.
        for i in range(1, N):
            running_max, running_argmax = -float('inf'), -1
            for t_idx in range(1, K):
                t, t_prev = T_v[t_idx], T_v[t_idx - 1]
                val = dp[i-1, t_idx - 1] + lambda_penalty * t_prev
                if val > running_max: running_max, running_argmax = val, t_idx - 1

                score_it = event_frame_scores[i].get(vid, {}).get(t, 0.0)

                # Cập nhật điểm số hàm mục tiêu bao gồm: Điểm tại t + Điểm lớn nhất tích lũy - Hệ số phạt khoảng cách (lambda_penalty).
                dp[i, t_idx] = score_it + running_max - lambda_penalty * t
                trace[i, t_idx] = running_argmax

        # Dò ngược (Backtracking) để tái tạo chuỗi khung hình tối ưu từ ma trận lưu vết.
        best_last_t_idx = np.argmax(dp[N-1])
        best_vid_score = dp[N-1, best_last_t_idx]

        if best_last_t_idx != -1 and best_vid_score > -float('inf'):
            seq, curr_t, valid = [], best_last_t_idx, True
            for i in range(N-1, -1, -1):
                if curr_t == -1: valid = False; break
                seq.append(T_v[curr_t])
                curr_t = trace[i, curr_t]

            if valid:
                seq.reverse()
                # Ánh xạ chỉ số nội bộ (n) sang chỉ số khung hình thực tế (frame_idx) của video thông qua từ điển map_db.
                mapped_seq = [map_db.get(vid, {}).get(n, n) for n in seq]
                final_sequences.append({'video_name': vid, 'score': best_vid_score, 'frames': mapped_seq, 'n_frames': seq})

    # Sắp xếp lại danh sách các chuỗi tìm được theo điểm số giảm dần và trả về kết quả Top 100.
    final_sequences.sort(key=lambda x: x['score'], reverse=True)
    return final_sequences[:100]

In [ ]:
# [BLOCK 4] RE-RANKING BẰNG VLM

import os, json
from PIL import Image

# Khai báo đường dẫn gốc trỏ tới kho dữ liệu khung hình (Keyframes) lưu trên Google Drive.
KEYFRAMES_DIR = "/content/drive/MyDrive/AIC26/Keyframes"

# Hàm tải ảnh tĩnh từ ổ đĩa.
# Sử dụng f"{int(n_val):04d}.jpg" để ép kiểu định dạng chuỗi 4 chữ số (ví dụ: 0001.jpg, 0123.jpg) nhằm đồng bộ tuyệt đối với chuẩn lưu trữ.
def load_frame_image(video_id, n_val):
    img_path = os.path.join(KEYFRAMES_DIR, video_id, f"{int(n_val):04d}.jpg")
    return Image.open(img_path) if os.path.exists(img_path) else None

def vlm_global_verifier(video_id, anchor_frames, sub_events):
    # Tải toàn bộ chuỗi hình ảnh tương ứng với chuỗi sự kiện.
    images = []
    for n in anchor_frames:
        img = load_frame_image(video_id, n)
        if img: images.append(img)

    if len(images) != len(sub_events): return 0.0

    prompt = f"""You are a RUTHLESS AI judge evaluating video retrieval. Look at these {len(images)} images in order.
    Do they EXACTLY match these events?
    """
    # Trình bày chuỗi sự kiện theo thứ tự thời gian.
    for idx, evt in enumerate(sub_events):
        prompt += f"Image {idx+1}: {evt}\n"

    # Áp đặt các quy tắc loại trừ nghiêm ngặt để hạn chế tối đa hiện tượng "ảo giác" của mô hình AI, bao gồm: kiểm tra nhận diện chữ (OCR), công cụ (Tools) và vật thể (Objects).
    prompt += """
    CRITICAL ELIMINATION RULES (Score 0.0 IMMEDIATELY if any rule is broken):
    1. OCR CHECK: Read ALL text/subtitles on the screen. If the text says "GIẤM" (vinegar) or something else, but the prompt asks for "NẤM" (mushroom), SCORE 0.0!
    2. TOOL CHECK: The action "Cắt" (cutting) REQUIRES a visible knife (dao) and usually a cutting board. If you see a person using a SPOON, a BLENDER, or pouring liquid, SCORE 0.0.
    3. OBJECT CHECK: You must clearly identify the specific object (mushroom, tofu, water chestnut).

    If it is a PERFECT match, score 1.0. Otherwise, score 0.0.
    Return ONLY JSON: {"confidence_score": <float>}
    """

    # Truy vấn mô hình Ngôn ngữ - Hình ảnh (VLM).
    try:
        res = client.models.generate_content(
            model='gemini-3.1-pro',
            contents=images + [prompt],
            # Ép mô hình phải trả về định dạng chuẩn JSON để thuật toán dễ dàng bóc tách điểm số.
            config=types.GenerateContentConfig(response_mime_type="application/json", temperature=0.0)
        )
        return float(json.loads(res.text.strip()).get('confidence_score', 0.0))
    except:
        return 0.0

In [ ]:
# [BLOCK 5] THỰC THI CHUỖI TỔNG THỂ & XUẤT KẾT QUẢ CSV

import os, shutil, csv, time
from google.colab import files

trake_queries = {
    # ------------------ TỪ ẢNH image_b7f1e6.png ------------------
    "query-p2-3-kis": "E1: Cận cảnh đầu bếp thả một loại rau củ có màu cam rực rỡ vào một nồi nước súp đang sôi sục.\nE2: Cảnh trút/đổ nguyên liệu dạng hình ống nhỏ (như nui/pasta) từ một chiếc đĩa xuống nồi.\nE3: Đầu bếp tiếp tục cho thêm các miếng rau củ màu xanh lá cây và các lát nấm có màu nâu vào nồi.\nE4: Khung hình cuối, đầu bếp trút nguyên một đĩa thịt (đã được thái thành nhiều lát mỏng) vào nồi để nấu chung với hỗn hợp trước đó.",

    "query-p2-4-kis": "E1: Khung hình có bố cục chia đôi (split-screen): Nửa bên phải là một giáo viên nữ đang ngồi giảng bài. Nửa bên trái là tấm bảng nền xanh lá (green board) hiển thị rõ dòng chữ Tiếng Anh của hai câu ví dụ (có chứa từ vựng liên quan đến bảo tàng hoặc bóng đá).\nE2: Trên bảng xanh xuất hiện thêm các khối văn bản/dòng chữ giải thích mới. Điểm chốt hạ: Ở dòng cuối cùng trên bảng hiện ra một công thức cấu trúc ngữ pháp Tiếng Anh có chứa các ký hiệu (như dấu cộng +, mũi tên, S, V).",

    # ------------------ TỪ ẢNH image_b7f207.png ------------------
    "query-p2-9-kis": "E1: Cảnh các người mẫu trình diễn thời trang, mặc trang phục phom dáng rộng màu kem. Điểm nhấn là trên vải có may đắp/chắp vá các mảng màu sắc sặc sỡ và họa tiết hình học.\nE2: Bối cảnh chuyển sang thảm cỏ xanh ngoài trời, nơi trưng bày nhiều đồ vật/tác phẩm nghệ thuật sặc sỡ, có rất đông người dân và du khách đang đứng tụ tập tham quan.\nE3: Góc máy quay cận cảnh (close-up) tĩnh vào các món đồ thủ công (hình búp bê và quả cầu). Nhìn rõ bề mặt đồ vật được chắp vá từ vô số mảnh vải vụn có màu sắc và hoa văn hoàn toàn khác nhau.",

    # ------------------ TỪ ẢNH image_b7f20b.png ------------------
    "query-p2-11-kis": "E1: Cảnh một người phụ nữ đang thao tác lấy một phần thành phẩm thức ăn ra khỏi một chiếc nồi (hoặc vật chứa) có màu đỏ nổi bật.\nE2: Góc quay cận cảnh vào món ăn thành phẩm: có màu trắng, hình thù nở phồng to, giống như nhiều sợi que/sợi dài dính chụm lại với nhau thành từng cụm.\nE3: Cảnh sắp xếp, đặt các cụm thành phẩm dạng sợi que màu trắng này lên trên bề mặt của một chiếc dĩa (hoặc vật liệu lót) cũng có màu trắng.",

    # ------------------ TỪ ẢNH image_b84389.png ------------------
    "query-p2-15-kis": "E1: Cảnh cận bàn tay đầu bếp đang trực tiếp nhào trộn, bóp một lớp bột ướt (sau khi sơ chế nước lạnh) dính chung với khối thực phẩm chính.\nE2: Cảnh quay thẳng vào một chiếc chảo có màu đỏ đang chứa dầu ăn. Trong chảo có sự tương phản màu sắc rõ rệt: xuất hiện chính xác 1 miếng nguyên liệu màu xanh lá cây và 2 miếng nguyên liệu màu vàng đang được chiên.",

    "query-p2-16-kis": "E1: Cảnh một người đang cầm đôi đũa liên tục khuấy, đảo các nguyên liệu (có màu đỏ pha lẫn chút màu xanh lá cây) bên trong một chiếc nồi thủy tinh trong suốt.\nE2: Động tác trút/đổ một lượng lớn chất lỏng vào nồi. Ngay sau đó là chuỗi thao tác dùng muỗng múc và nêm liên tục nhiều lần các loại gia vị (dạng bột màu trắng và dạng hạt) vào trong nồi nước dùng.",

    "query-p2-17-kis": "E1: Cận cảnh khối nguyên liệu (đã được chiên sơ qua) bị dao cắt xẻ dọc làm 2 nửa, nhưng phần đáy/cuống vẫn dính liền nhau chứ không bị đứt rời hoàn toàn.\nE2: Người đàn ông dùng dụng cụ (như thìa/dao) luồn vào trong để cạo, lách và tách rời toàn bộ phần lõi bên trong ra khỏi lớp vỏ ngoài.\nE3: Đặt phần lõi nguyên liệu vừa được nạo ra lên trên một đồ vật/dụng cụ (như khay, thớt hoặc dĩa) có màu trắng.",

    # ------------------ TỪ ẢNH image_b843a8.png ------------------
    "query-p2-19-kis": "E1: Con múa lân màu vàng đang biểu diễn trên hệ thống cọc trụ (Mai Hoa Thung). Lân chụm đứng thăng bằng trên 3 cọc trụ, sau đó vươn khéo léo đặt nốt chiếc chân còn lại lên đỉnh trụ thứ tư.\nE2: Đầu lân vàng cúi gập xuống ngoạm/ngậm lấy một cành hoa có màu tím, ngay lập tức thực hiện động tác xoay toàn bộ thân người quay lưng lại.\nE3: Phần đầu lân thực hiện động tác gật, mở và đóng hàm múa liên tục theo nhịp, kết thúc bằng việc nhả cành hoa màu tím rơi tự do xuống đất.",

    # ------------------ TỪ ẢNH image_b843c1.png ------------------
    "query-p2-26-kis": "E1: Bảng trình chiếu hiển thị bản vẽ đồ họa hình học không gian: Một đường thẳng xiên đâm xuyên qua một mặt phẳng, bên cạnh là hai mặt phẳng đang cắt/giao nhau.\nE2: Bản vẽ thay đổi: Xuất hiện một mặt phẳng và một điểm lơ lửng phía trên. Có một đoạn thẳng được vẽ nối thẳng đứng từ điểm đó vuông góc xuống mặt phẳng.\nE3: Màn hình hiển thị một khối chóp/tứ diện có 4 đỉnh. Đồ họa bắt đầu vẽ bổ sung thêm các đường nét đứt, các điểm chấm và ký hiệu góc vuông vào hình.",

    # ------------------ TỪ ẢNH image_b843c6.png ------------------
    "query-p2-30-kis": "E1: Cảnh một người đang vung tay phất mạnh một lá cờ báo hiệu tại vạch đích của đường đua xe đạp, bối cảnh phía sau có thể nhìn thấy mặt hồ nước.\nE2: Góc máy quay rộng toàn cảnh một đoàn rất đông tay đua xe đạp đang nghiêng người ôm cua trên một con đường phố rợp bóng cây xanh.\nE3: Cận cảnh một khán giả nam đứng lề đường với tư thế cực kỳ đặc trưng: tay trái cầm điện thoại di động hướng lên, tay phải vươn ra cầm một cây gậy dài (gậy selfie/gimbal) chĩa thẳng camera về phía đoàn đua.",

    # ------------------ TỪ ẢNH image_b843c9.png ------------------
    "query-p2-32-kis": "E1: Cảnh hai mẹ con đang cầm và cùng nhau nhìn chăm chú vào màn hình một chiếc điện thoại di động để trò chuyện/video call.\nE2: Chuyển sang cảnh người con đang đứng trả lời phỏng vấn. Rất quan trọng (Mỏ neo không gian): Ở bối cảnh nền phía sau lưng người con, có hình bóng của một người phụ nữ da màu (da đen) đang bước đi ngang qua khung hình."
}
# Khởi tạo môi trường lưu trữ kết quả.
# Hệ thống sẽ tự động dọn dẹp thư mục cũ (nếu có) để tránh xung đột dữ liệu (data conflict) từ các lần chạy thực nghiệm trước đó.
SUBMISSION_DIR = "trake_submission"
if os.path.exists(SUBMISSION_DIR): shutil.rmtree(SUBMISSION_DIR)
os.makedirs(SUBMISSION_DIR, exist_ok=True)

# Khai báo các siêu tha# Kích hoạt thuật toán Quy hoạch động để trích xuất danh sách 100 video số (Hyperparameters) cho quy trình Hậu xử lý (Post-processing).
USE_VLM = True               # Bật/tắt cơ chế Giám khảo AI (Global Verifier).
TOP_K_VLM_CANDIDATES = 30    # Chỉ giới hạn chấm điểm VLM cho top 30 video tiềm năng nhất để tối ưu thời gian.

# Vòng lặp thực thi cốt lõi: Xử lý tuần tự từng truy vấn có trong bộ dữ liệu.
for q_id, q_text in trake_queries.items():

    # Kích hoạt thuật toán Quy hoạch động để trích xuất danh sách 100 video có điểm số không gian gần nhất.
    dp_results = dp_solve(q_id, q_text, top_videos=100, lambda_penalty=0.00005)
    sub_events = split_trake_query(q_text)
    final_list = dp_results.copy()

    # Tiến trình Đánh giá lại (Re-ranking) bằng VLM (nếu USE_VLM = True).
    if USE_VLM and len(final_list) > 0:
        for rank, candidate in enumerate(final_list[:TOP_K_VLM_CANDIDATES]):
            vid = candidate['video_name']

            # Trích xuất chuỗi khung hình nội bộ (n_frames) để tải ảnh tương ứng.
            n_seq = candidate.get('n_frames', candidate.get('frames', []))

            # Gọi mô hình VLM chấm điểm độ khớp ngữ nghĩa thực tế.
            vlm_score = vlm_global_verifier(vid, n_seq, sub_events)
            candidate['vlm_score'] = vlm_score

            # Cơ chế Dừng sớm (Early Stopping): Nếu phát hiện video khớp hoàn hảo (điểm >= 0.95), ngắt vòng lặp ngay lập tức để bảo toàn kết quả và tiết kiệm API Quota.
            if vlm_score >= 0.95:
                break

            # Ngủ 2 giây để tránh lỗi vượt quá giới hạn tần suất yêu cầu (Rate Limit) của máy chủ API.
            time.sleep(2)

        # Cập nhật lại thứ hạng (Re-rank) cho tập TOP_K ứng viên dựa trên điểm số VLM mới, các video thuộc top sau (từ 31-100) vẫn giữ nguyên vị trí cũ.
        top_k_part = final_list[:TOP_K_VLM_CANDIDATES]
        top_k_part.sort(key=lambda x: x.get('vlm_score', -1.0), reverse=True)
        final_list = top_k_part + final_list[TOP_K_VLM_CANDIDATES:]

    # Ghi xuất kết quả cuối cùng (Top 100) ra tệp định dạng CSV.
    if len(final_list) > 0:
        with open(f"{SUBMISSION_DIR}/{q_id}.csv", 'w', newline='') as f:
            writer = csv.writer(f)
            for res in final_list[:100]:
                writer.writerow([res['video_name']] + res['frames'])

# Đóng gói (Archive) toàn bộ thư mục CSV thành một tệp nén (.zip) duy nhất.
shutil.make_archive("submission", 'zip', root_dir=SUBMISSION_DIR)

# Kích hoạt lệnh giao tiếp với trình duyệt để tự động tải tệp nén xuống hệ thống cục bộ.
files.download("submission.zip")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
# [BLOCK 6] GIAO DIỆN KIỂM TRA TRỰC QUAN

# 1. Cài đặt thư viện Gradio ở chế độ ẩn (-q) để xây dựng giao diện người dùng (UI) trên nền web.
!pip install -q gradio
import gradio as gr
import os, csv
from PIL import Image

# 2. Khai báo các đường dẫn thư mục gốc chứa bộ dữ liệu khung hình (Keyframes) trên bộ nhớ Drive và thư mục chứa kết quả truy xuất (CSV).
BASE_KF_DIR = "/content/drive/MyDrive/AIC26/Keyframes"
SUBMISSION_DIR = "trake_submission"

# 3. Xây dựng cấu trúc dữ liệu bản đồ đường dẫn (Path Mapping) cho toàn bộ tập video.
vid_path_map = {}
if os.path.exists(BASE_KF_DIR):
    # Thuật toán quét sâu (Deep scan) qua các thư mục con phân mảnh để ánh xạ mã định danh video (Video ID) tới đường dẫn vật lý thực tế.
    for sub_kf in os.listdir(BASE_KF_DIR):
        sub_kf_path = os.path.join(BASE_KF_DIR, sub_kf)
        if os.path.isdir(sub_kf_path):
            for vid_folder in os.listdir(sub_kf_path):
                vid_path_map[vid_folder.upper()] = os.path.join(sub_kf_path, vid_folder)
    print(f"✅ Đã lập bản đồ thành công {len(vid_path_map)} video!")
else:
    print(f"❌ KHÔNG TÌM THẤY THƯ MỤC MẸ: {BASE_KF_DIR}")

# 4. Khởi tạo từ điển nghịch đảo (Reverse Dictionary) để giải mã chỉ số khung hình.
# Chuyển đổi ngược từ chỉ số khung hình thực tế (frame_idx) về chỉ số thứ tự nội bộ (n) để hệ thống định vị chính xác tên tệp hình ảnh đuôi .jpg.
reverse_map_db = {}
if 'map_db' in globals():
    for vid, mapping in map_db.items():
        reverse_map_db[vid] = {v: k for k, v in mapping.items()}

# 5. Hàm cốt lõi xử lý logic trực quan hóa truy vấn (Visualization Logic).
def visualize_query(query_id):
    csv_path = os.path.join(SUBMISSION_DIR, f"{query_id}.csv")
    if not os.path.exists(csv_path):
        return None, f"❌ LỖI: Không tìm thấy file {csv_path}."

    gallery_images = []
    log_msgs = []

    try:
        # Mở và phân tích cú pháp tệp CSV chứa kết quả truy xuất.
        with open(csv_path, 'r') as f:
            reader = csv.reader(f)
            rows = list(reader)

            if not rows:
                return None, "❌ LỖI: File CSV này trống rỗng."

            # Vòng lặp duyệt qua các bản ghi, giới hạn trích xuất ở 10 kết quả (Top 10) có điểm số cao nhất để tối ưu hiệu suất kết xuất (rendering).
            for rank, row in enumerate(rows):
                if rank >= 10: break
                if not row: continue

                vid = row[0].strip().upper()
                frames = row[1:]

                # Truy vấn đường dẫn gốc của video từ bản đồ thư mục (vid_path_map).
                vid_dir = vid_path_map.get(vid)

                for event_idx, frame_str in enumerate(frames):
                    real_frame = int(float(frame_str))

                    # Nội suy số thứ tự ảnh (n_val) từ frame gốc, sau đó ép định dạng 4 chữ số (0001.jpg, 0123.jpg).
                    n_val = reverse_map_db.get(vid, {}).get(real_frame, real_frame)
                    img_name = f"{int(n_val):04d}.jpg"

                    # Xử lý hình ảnh hợp lệ.
                    if vid_dir:
                        img_path = os.path.join(vid_dir, img_name)
                        if os.path.exists(img_path):
                            img = Image.open(img_path)
                            caption = f"Rank {rank+1} | E{event_idx+1}\n{vid}\nFrame: {real_frame} -> {img_name}"
                            gallery_images.append((img, caption))
                        # Cơ chế xử lý lỗi khi mất tệp hình ảnh (Image File Missing): Tạo khối ảnh đỏ cảnh báo.
                        else:
                            blank_img = Image.new('RGB', (300, 200), color=(220, 53, 69))
                            caption = f"❌ MẤT ẢNH\n{vid}\n{img_name}"
                            gallery_images.append((blank_img, caption))
                            log_msgs.append(f"Thiếu file ảnh: {img_path}")
                    # Cơ chế xử lý lỗi khi mất thư mục (Directory Missing): Tạo khối ảnh đỏ cảnh báo.
                    else:
                        blank_img = Image.new('RGB', (300, 200), color=(220, 53, 69))
                        caption = f"❌ MẤT FOLDER\n{vid}"
                        gallery_images.append((blank_img, caption))
                        log_msgs.append(f"Không tìm thấy thư mục chứa video: {vid}")

        # Trả về mảng hình ảnh kèm theo trạng thái chẩn đoán hệ thống.
        status = "✅ Tải ảnh thành công!"
        if log_msgs:
            status += "\n⚠️ Các cảnh báo:\n" + "\n".join(log_msgs)

        return gallery_images, status

    except Exception as e:
        return None, f"❌ LỖI HỆ THỐNG: {str(e)}"

# 6. Khởi tạo và cấu hình Giao diện Người dùng Đồ họa (GUI) thông qua nền tảng Gradio.
with gr.Blocks(theme=gr.themes.Soft()) as demo:
    gr.Markdown(f"## 👁️ AI Challenge Visualizer - Team TayLor")

    with gr.Row():
        query_list = list(trake_queries.keys()) if 'trake_queries' in globals() else []
        query_dropdown = gr.Dropdown(choices=query_list, label="Chọn Query ID", value=query_list[0] if query_list else None)
        btn_load = gr.Button("Tải Ảnh", variant="primary")

    status_log = gr.Textbox(label="Trạng thái hệ thống", lines=4)
    gallery = gr.Gallery(
        label="Top 10 Kết quả (Hiển thị theo chuỗi sự kiện E1, E2...)",
        show_label=True,
        elem_id="gallery",
        columns=4,
        height="auto"
    )

    # Gắn sự kiện (Event Listener) cho nút bấm, liên kết với hàm visualize_query.
    btn_load.click(fn=visualize_query, inputs=query_dropdown, outputs=[gallery, status_log])

# 7. Khởi chạy máy chủ web cục bộ và tạo đường dẫn chia sẻ công khai (Public URL).
demo.launch(debug=True, share=True)

✅ Đã lập bản đồ thành công 875 video!


/tmp/ipykernel_1187/3536038189.py:100: UserWarning: The parameters have been moved from the Blocks constructor to the launch() method in Gradio 6.0: theme. Please pass these parameters to launch() instead.
  with gr.Blocks(theme=gr.themes.Soft()) as demo:


Colab notebook detected. This cell will run indefinitely so that you can see errors and logs. To turn off, set debug=False in launch().
* Running on public URL: https://e79572b6b638a26cfe.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


AttributeError: module 'gradio' has no attribute 'blocks'